# 🎓 Predicción de Distritos Escolares en Riesgo
## Proyecto Final - Datos Masivos | Amazon EMR

---

**Objetivo:** Sistema de alerta temprana para identificar distritos escolares con graduación < 80%

**Entorno:** Amazon EMR + Apache Spark + S3

---

### Contenido:
1. Introducción y Objetivo
2. Descripción de Datasets
3. Configuración Spark EMR
4. Carga de Datos desde S3
5. Análisis Exploratorio (EDA)
6. Limpieza de Datos
7. Integración de Datasets
8. Análisis Detallado (10 Puntos)
9. Propuesta de Valor
10. Conclusiones

---
## 1. Introducción y Objetivo

### Contexto
El Estado de Nueva York tiene disparidades significativas en tasas de graduación escolar. Este proyecto utiliza **Big Data con Apache Spark en Amazon EMR** para identificar distritos en riesgo.

### Objetivo
1. Analizar tasas de graduación de todos los distritos de NY
2. Integrar datos socioeconómicos del Census Bureau
3. Identificar factores de riesgo
4. Proponer sistema de alerta temprana

### Stack Tecnológico
- **Amazon EMR** - Cluster administrado de Spark
- **Amazon S3** - Almacenamiento de datos
- **Apache Spark 3.x** - Procesamiento distribuido
- **PySpark** - API Python para Spark

---
## 2. Descripción de Datasets

### Dataset Principal: Tasas de Graduación NY (2021)
| Atributo | Valor |
|----------|-------|
| **Fuente** | NY State Education Department |
| **Archivo** | GRAD_RATE_AND_OUTCOMES_2021.csv |
| **Registros** | 220,304 |
| **Columnas** | 36 |
| **Variables clave** | grad_pct, dropout_pct, county_name, lea_name |

### Dataset Complementario: US Census (2017)
| Atributo | Valor |
|----------|-------|
| **Fuente** | US Census Bureau - American Community Survey |
| **Archivo** | acs2017_county_data.csv |
| **Registros** | 3,220 (62 condados NY) |
| **Variables clave** | Income, Poverty, ChildPoverty, Unemployment |

### Justificación del Dataset Complementario
- La pobreza infantil está correlacionada con resultados educativos
- JOIN natural por `county_name`
- Permite contextualizar rendimiento escolar

---
## 3. Configuración Spark en EMR

In [ ]:
# Importaciones
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
import matplotlib.pyplot as plt
import pandas as pd

print("✅ Librerías importadas")

In [ ]:
# Inicializar SparkSession para EMR
# En EMR, Spark ya está preconfigurado con acceso a S3
spark = SparkSession.builder \
    .appName("DistritoEnRiesgo_EMR") \
    .enableHiveSupport() \
    .getOrCreate()

# Configuración para S3
spark.sparkContext.setLogLevel("WARN")

print("="*60)
print("🚀 SPARK SESSION EN AMAZON EMR")
print("="*60)
print(f"Versión Spark: {spark.version}")
print(f"App Name: {spark.sparkContext.appName}")
print(f"Master: {spark.sparkContext.master}")

In [ ]:
# ⚠️ CONFIGURAR TU BUCKET S3 AQUÍ
S3_BUCKET = "s3://datos-masivos-jjimenez-2024"  # CAMBIAR POR TU BUCKET

# Rutas de los datasets en S3
PATH_GRAD_RATE = f"{S3_BUCKET}/data/GRAD_RATE_AND_OUTCOMES_2021.csv"
PATH_CENSUS = f"{S3_BUCKET}/data/acs2017_county_data.csv"
PATH_OUTPUT = f"{S3_BUCKET}/output/"

print(f"📁 Bucket S3: {S3_BUCKET}")
print(f"📁 Dataset Principal: {PATH_GRAD_RATE}")
print(f"📁 Dataset Complementario: {PATH_CENSUS}")
print(f"📁 Output: {PATH_OUTPUT}")

---
## 4. Carga de Datos desde S3

In [ ]:
# Cargar Dataset Principal: Tasas de Graduación
print("📥 Cargando dataset de graduación desde S3...")

df_grad = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("quote", '"') \
    .option("escape", '"') \
    .csv(PATH_GRAD_RATE)

# Cache para mejor rendimiento
df_grad.cache()

print(f"\n✅ DATASET PRINCIPAL CARGADO")
print(f"   Registros: {df_grad.count():,}")
print(f"   Columnas: {len(df_grad.columns)}")
print(f"   Particiones: {df_grad.rdd.getNumPartitions()}")

In [ ]:
# Cargar Dataset Complementario: Census
print("📥 Cargando dataset Census desde S3...")

df_census = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(PATH_CENSUS)

print(f"\n✅ DATASET COMPLEMENTARIO CARGADO")
print(f"   Registros: {df_census.count():,}")
print(f"   Columnas: {len(df_census.columns)}")

In [ ]:
# Schema del dataset principal
print("📋 SCHEMA - Dataset Graduación:")
df_grad.printSchema()

In [ ]:
# Muestra de datos
print("📋 MUESTRA DE DATOS:")
df_grad.select("lea_name", "county_name", "grad_pct", "dropout_pct", "subgroup_name").show(5, truncate=False)

---
## 5. Análisis Exploratorio de Datos (EDA)

In [ ]:
print("🔍 ANÁLISIS DE VALORES NULOS")
print("="*60)

total_rows = df_grad.count()

# Columnas clave para analizar
cols_to_check = ["grad_pct", "dropout_pct", "county_name", "lea_name", 
                 "aggregation_type", "subgroup_name", "enroll_cnt"]

print(f"{'Columna':<25} {'Nulos':>10} {'%':>10}")
print("-"*50)

for col_name in cols_to_check:
    null_count = df_grad.filter(
        (col(col_name).isNull()) | 
        (col(col_name) == "") | 
        (col(col_name) == "s")
    ).count()
    null_pct = (null_count / total_rows) * 100
    print(f"{col_name:<25} {null_count:>10,} {null_pct:>9.1f}%")

In [ ]:
print("\n📊 TIPOS DE AGREGACIÓN:")
df_grad.groupBy("aggregation_type") \
    .agg(count("*").alias("cantidad")) \
    .orderBy(desc("cantidad")) \
    .show()

In [ ]:
print("\n📊 SUBGRUPOS DEMOGRÁFICOS:")
df_grad.groupBy("subgroup_name") \
    .agg(count("*").alias("cantidad")) \
    .orderBy(desc("cantidad")) \
    .show(10)

---
## 6. Limpieza de Datos

### Problemas Identificados:
| Problema | Solución |
|----------|----------|
| Múltiples niveles (State, District, School) | Filtrar solo District |
| Porcentajes como string con '%' | Convertir a Double |
| Valores suprimidos ('s') | Convertir a NULL |
| Múltiples subgrupos | Filtrar solo 'All Students' |
| Outliers | Detectar con IQR, marcar pero no eliminar |

In [ ]:
print("🧹 LIMPIEZA DE DATOS")
print("="*60)

# Paso 1: Filtrar Distritos y All Students
df_clean = df_grad \
    .filter(col("aggregation_type") == "District") \
    .filter(col("subgroup_name") == "All Students")

print(f"✅ Paso 1: Filtrar District + All Students")
print(f"   Registros: {df_clean.count():,}")

In [ ]:
# Paso 2: Convertir porcentajes a numérico
df_clean = df_clean \
    .withColumn("grad_pct_clean", 
        when(col("grad_pct").contains("%"), 
             regexp_replace(col("grad_pct"), "%", "").cast(DoubleType()))
        .otherwise(None)) \
    .withColumn("dropout_pct_clean",
        when(col("dropout_pct").contains("%"),
             regexp_replace(col("dropout_pct"), "%", "").cast(DoubleType()))
        .otherwise(None)) \
    .withColumn("local_pct_clean",
        when(col("local_pct").contains("%"),
             regexp_replace(col("local_pct"), "%", "").cast(DoubleType()))
        .otherwise(None)) \
    .withColumn("reg_adv_pct_clean",
        when(col("reg_adv_pct").contains("%"),
             regexp_replace(col("reg_adv_pct"), "%", "").cast(DoubleType()))
        .otherwise(None)) \
    .withColumn("still_enr_pct_clean",
        when(col("still_enr_pct").contains("%"),
             regexp_replace(col("still_enr_pct"), "%", "").cast(DoubleType()))
        .otherwise(None))

print(f"✅ Paso 2: Convertir porcentajes string → Double")

In [ ]:
# Paso 3: Eliminar registros sin tasa de graduación
df_clean = df_clean.filter(col("grad_pct_clean").isNotNull())

print(f"✅ Paso 3: Eliminar registros sin grad_pct")
print(f"   Registros: {df_clean.count():,}")

In [ ]:
# Paso 4: Crear variable target
df_clean = df_clean.withColumn("en_riesgo", 
    when(col("grad_pct_clean") < 80, 1).otherwise(0))

print(f"✅ Paso 4: Crear target (en_riesgo = 1 si grad < 80%)")

# Distribución del target
print("\n📊 Distribución del target:")
df_clean.groupBy("en_riesgo") \
    .agg(
        count("*").alias("cantidad"),
        round(avg("grad_pct_clean"), 2).alias("grad_promedio")
    ).show()

In [ ]:
# Paso 5: Verificar duplicados
duplicados = df_clean \
    .groupBy("lea_name", "county_name") \
    .count() \
    .filter(col("count") > 1) \
    .count()

print(f"✅ Paso 5: Verificar duplicados")
print(f"   Duplicados: {duplicados}")

In [ ]:
# Paso 6: Detección de Outliers (IQR)
print("\n🔍 DETECCIÓN DE OUTLIERS (IQR)")
print("-"*50)

quantiles = df_clean.approxQuantile("grad_pct_clean", [0.25, 0.5, 0.75], 0.01)
q1, median, q3 = quantiles[0], quantiles[1], quantiles[2]
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

print(f"   Q1 (25%):          {q1:.2f}%")
print(f"   Mediana (50%):     {median:.2f}%")
print(f"   Q3 (75%):          {q3:.2f}%")
print(f"   IQR:               {iqr:.2f}")
print(f"   Límite inferior:   {lower_bound:.2f}%")
print(f"   Límite superior:   {upper_bound:.2f}%")

outliers_count = df_clean.filter(
    (col("grad_pct_clean") < lower_bound) | 
    (col("grad_pct_clean") > upper_bound)
).count()

print(f"\n⚠️  Outliers detectados: {outliers_count} ({outliers_count/df_clean.count()*100:.1f}%)")

# Marcar outliers (no eliminar - son importantes)
df_clean = df_clean.withColumn("is_outlier",
    when((col("grad_pct_clean") < lower_bound) | 
         (col("grad_pct_clean") > upper_bound), 1).otherwise(0))

print("✅ Paso 6: Outliers marcados (no eliminados)")

In [ ]:
# Resumen de limpieza
print("\n" + "="*60)
print("📋 RESUMEN DEL PROCESO DE LIMPIEZA")
print("="*60)

print(f"""
| Problema                        | Solución                           |
|---------------------------------|------------------------------------|
| Múltiples niveles agregación    | Filtrar solo 'District'            |
| Múltiples subgrupos             | Filtrar solo 'All Students'        |
| Porcentajes como string         | Convertir a Double                 |
| Valores suprimidos ('s')        | Convertir a NULL                   |
| Registros sin grad_pct          | Eliminar                           |
| Outliers                        | Marcar (importantes para análisis) |

📊 RESULTADO:
   Registros finales: {df_clean.count():,}
   Distritos únicos:  {df_clean.select('lea_name').distinct().count()}
   Condados únicos:   {df_clean.select('county_name').distinct().count()}
""")

---
## 7. Integración de Datasets

In [ ]:
print("🔗 INTEGRACIÓN DE DATASETS")
print("="*60)

# Preparar Census: filtrar NY y limpiar nombre de condado
df_census_ny = df_census \
    .filter(col("State") == "New York") \
    .withColumn("county_name", regexp_replace(col("County"), " County", "")) \
    .select(
        col("county_name"),
        col("TotalPop"),
        col("Income"),
        col("IncomePerCap"),
        col("Poverty"),
        col("ChildPoverty"),
        col("Unemployment"),
        col("Professional"),
        col("White"),
        col("Black"),
        col("Hispanic"),
        col("Asian")
    )

print(f"✅ Condados de NY en Census: {df_census_ny.count()}")
print("\n📋 Muestra Census NY:")
df_census_ny.show(5)

In [ ]:
# Realizar JOIN por county_name
df_merged = df_clean.join(df_census_ny, on="county_name", how="left")

# Cache para análisis
df_merged.cache()

print(f"✅ JOIN completado")
print(f"   Registros totales: {df_merged.count():,}")

# Verificar éxito del JOIN
con_census = df_merged.filter(col("Income").isNotNull()).count()
sin_census = df_merged.filter(col("Income").isNull()).count()

print(f"\n✅ Con datos socioeconómicos: {con_census:,}")
print(f"⚠️  Sin match en Census: {sin_census:,}")

In [ ]:
# Condados sin match (si los hay)
if sin_census > 0:
    print("\n📋 Condados sin datos socioeconómicos:")
    df_merged.filter(col("Income").isNull()) \
        .select("county_name").distinct().show()

In [ ]:
# Dataset final para análisis
df_final = df_merged.select(
    # Identificadores
    col("lea_name"),
    col("county_name"),
    col("nyc_ind"),
    
    # Target y flags
    col("grad_pct_clean"),
    col("en_riesgo"),
    col("is_outlier"),
    
    # Features educativas
    col("dropout_pct_clean"),
    col("still_enr_pct_clean"),
    col("local_pct_clean"),
    col("reg_adv_pct_clean"),
    col("enroll_cnt"),
    
    # Features socioeconómicas
    col("TotalPop"),
    col("Income"),
    col("Poverty"),
    col("ChildPoverty"),
    col("Unemployment"),
    col("Professional"),
    col("White"),
    col("Black"),
    col("Hispanic")
)

df_final.cache()

print(f"\n📋 DATASET FINAL PARA ANÁLISIS")
print(f"   Registros: {df_final.count():,}")
print(f"   Columnas: {len(df_final.columns)}")
print("\n📋 Schema:")
df_final.printSchema()

---
## 8. Análisis Detallado (10 Puntos)

In [ ]:
print("="*70)
print("📊 ANÁLISIS 1: Distribución de Tasas de Graduación")
print("="*70)

# Estadísticas descriptivas
print("\nEstadísticas descriptivas:")
df_final.describe("grad_pct_clean").show()

# Distribución por rangos
print("\nDistribución por rangos:")
df_final.withColumn("rango_graduacion",
    when(col("grad_pct_clean") < 60, "1. Crítico (<60%)")
    .when(col("grad_pct_clean") < 70, "2. Muy bajo (60-70%)")
    .when(col("grad_pct_clean") < 80, "3. Bajo (70-80%)")
    .when(col("grad_pct_clean") < 90, "4. Medio (80-90%)")
    .otherwise("5. Alto (>90%)")) \
    .groupBy("rango_graduacion") \
    .agg(
        count("*").alias("cantidad"),
        round(avg("grad_pct_clean"), 2).alias("grad_promedio")
    ) \
    .orderBy("rango_graduacion") \
    .show()

In [ ]:
print("="*70)
print("📊 ANÁLISIS 2: NYC vs Resto del Estado")
print("="*70)

df_final.groupBy("nyc_ind") \
    .agg(
        count("*").alias("distritos"),
        round(avg("grad_pct_clean"), 2).alias("grad_promedio"),
        round(avg("dropout_pct_clean"), 2).alias("dropout_promedio"),
        round((sum(col("en_riesgo").cast("double")) / count("*") * 100), 2).alias("pct_en_riesgo")
    ) \
    .withColumn("ubicacion", when(col("nyc_ind") == 1, "NYC").otherwise("Resto NY")) \
    .select("ubicacion", "distritos", "grad_promedio", "dropout_promedio", "pct_en_riesgo") \
    .show()

In [ ]:
print("="*70)
print("📊 ANÁLISIS 3: Top 10 Condados con Mayor Riesgo")
print("="*70)

df_final.groupBy("county_name") \
    .agg(
        count("*").alias("distritos"),
        round(avg("grad_pct_clean"), 2).alias("grad_promedio"),
        sum("en_riesgo").alias("distritos_riesgo"),
        round((sum(col("en_riesgo").cast("double")) / count("*") * 100), 2).alias("pct_riesgo")
    ) \
    .filter(col("distritos") >= 3) \
    .orderBy(desc("pct_riesgo")) \
    .limit(10) \
    .show(truncate=False)

In [ ]:
print("="*70)
print("📊 ANÁLISIS 4: Correlaciones con Tasa de Graduación")
print("="*70)

# Correlaciones educativas
corr_dropout = df_final.stat.corr("dropout_pct_clean", "grad_pct_clean")
corr_still = df_final.stat.corr("still_enr_pct_clean", "grad_pct_clean")
corr_regadv = df_final.stat.corr("reg_adv_pct_clean", "grad_pct_clean")

print(f"""
CORRELACIONES EDUCATIVAS:
  Dropout %         →  Graduación: {corr_dropout:+.4f} {'⬇️ NEGATIVA' if corr_dropout < 0 else '⬆️ POSITIVA'}
  Still Enrolled %  →  Graduación: {corr_still:+.4f} {'⬇️ NEGATIVA' if corr_still < 0 else '⬆️ POSITIVA'}
  Regents Adv %     →  Graduación: {corr_regadv:+.4f} {'⬇️ NEGATIVA' if corr_regadv < 0 else '⬆️ POSITIVA'}

INTERPRETACIÓN:
  → Alta deserción = Menor graduación
  → Más "still enrolled" = Menor graduación a tiempo
  → Mejor Regents Advanced = Mayor graduación
""")

In [ ]:
print("="*70)
print("📊 ANÁLISIS 5: Brechas Demográficas")
print("="*70)

print("\nComposición demográfica por estado de riesgo:")
df_final.groupBy("en_riesgo") \
    .agg(
        round(avg("White"), 2).alias("pct_white"),
        round(avg("Black"), 2).alias("pct_black"),
        round(avg("Hispanic"), 2).alias("pct_hispanic")
    ) \
    .withColumn("estado", when(col("en_riesgo") == 1, "En Riesgo").otherwise("No Riesgo")) \
    .select("estado", "pct_white", "pct_black", "pct_hispanic") \
    .show()

In [ ]:
print("="*70)
print("📊 ANÁLISIS 6: Impacto de Pobreza en Graduación")
print("="*70)

df_with_poverty = df_final.filter(col("Poverty").isNotNull())

corr_poverty = df_with_poverty.stat.corr("Poverty", "grad_pct_clean")
corr_child_poverty = df_with_poverty.stat.corr("ChildPoverty", "grad_pct_clean")

print(f"""
CORRELACIONES SOCIOECONÓMICAS:
  Pobreza general   →  Graduación: {corr_poverty:+.4f}
  Pobreza infantil  →  Graduación: {corr_child_poverty:+.4f}

⚠️ HALLAZGO CLAVE: Mayor pobreza = Menor graduación
""")

print("\nGraduación por nivel de pobreza del condado:")
df_with_poverty.withColumn("nivel_pobreza",
    when(col("Poverty") < 10, "1. Baja (<10%)")
    .when(col("Poverty") < 15, "2. Media (10-15%)")
    .when(col("Poverty") < 20, "3. Alta (15-20%)")
    .otherwise("4. Muy Alta (>20%)")) \
    .groupBy("nivel_pobreza") \
    .agg(
        count("*").alias("distritos"),
        round(avg("grad_pct_clean"), 2).alias("grad_promedio"),
        round((sum(col("en_riesgo").cast("double")) / count("*") * 100), 2).alias("pct_riesgo")
    ) \
    .orderBy("nivel_pobreza") \
    .show()

In [ ]:
print("="*70)
print("📊 ANÁLISIS 7: Relación Ingreso vs Graduación")
print("="*70)

df_with_income = df_final.filter(col("Income").isNotNull())
corr_income = df_with_income.stat.corr("Income", "grad_pct_clean")

print(f"\nCorrelación Ingreso-Graduación: {corr_income:+.4f}")

print("\nGraduación por nivel de ingreso del condado:")
df_with_income.withColumn("nivel_ingreso",
    when(col("Income") < 50000, "1. Bajo (<$50K)")
    .when(col("Income") < 70000, "2. Medio ($50K-$70K)")
    .when(col("Income") < 90000, "3. Alto ($70K-$90K)")
    .otherwise("4. Muy Alto (>$90K)")) \
    .groupBy("nivel_ingreso") \
    .agg(
        count("*").alias("distritos"),
        round(avg("grad_pct_clean"), 2).alias("grad_promedio"),
        round(avg("Poverty"), 2).alias("pobreza_promedio")
    ) \
    .orderBy("nivel_ingreso") \
    .show()

In [ ]:
print("="*70)
print("📊 ANÁLISIS 8: Distritos en Situación Crítica (Outliers)")
print("="*70)

print("\n🚨 Distritos con graduación más baja:")
df_final.filter(col("is_outlier") == 1) \
    .orderBy("grad_pct_clean") \
    .select("lea_name", "county_name", "grad_pct_clean", 
            "dropout_pct_clean", "Poverty", "ChildPoverty") \
    .show(10, truncate=False)

In [ ]:
print("="*70)
print("📊 ANÁLISIS 9: Segmentación de Distritos por Nivel de Riesgo")
print("="*70)

df_segmented = df_final.withColumn("segmento",
    when(col("grad_pct_clean") < 70, "1. CRÍTICO - Intervención urgente")
    .when(col("grad_pct_clean") < 80, "2. RIESGO - Monitoreo intensivo")
    .when(col("grad_pct_clean") < 90, "3. ATENCIÓN - Seguimiento regular")
    .otherwise("4. ESTABLE - Mantener estrategia"))

print("\n📋 Distribución de segmentos:")
df_segmented.groupBy("segmento") \
    .agg(
        count("*").alias("distritos"),
        round(avg("grad_pct_clean"), 2).alias("grad_promedio"),
        round(avg("dropout_pct_clean"), 2).alias("dropout_promedio"),
        round(avg("Poverty"), 2).alias("pobreza_promedio")
    ) \
    .orderBy("segmento") \
    .show(truncate=False)

In [ ]:
print("="*70)
print("📊 ANÁLISIS 10: Perfil Comparativo Multivariado")
print("="*70)

print("\n📋 Perfil: Distritos EN RIESGO vs NO RIESGO")
df_final.groupBy("en_riesgo") \
    .agg(
        count("*").alias("n"),
        round(avg("grad_pct_clean"), 2).alias("graduacion"),
        round(avg("dropout_pct_clean"), 2).alias("desercion"),
        round(avg("Income"), 0).alias("ingreso_med"),
        round(avg("Poverty"), 2).alias("pobreza"),
        round(avg("ChildPoverty"), 2).alias("pobreza_inf"),
        round(avg("Unemployment"), 2).alias("desempleo")
    ) \
    .withColumn("estado", when(col("en_riesgo") == 1, "EN RIESGO").otherwise("NO RIESGO")) \
    .select("estado", "n", "graduacion", "desercion", "ingreso_med", 
            "pobreza", "pobreza_inf", "desempleo") \
    .show()

---
## 9. Propuesta de Valor

In [ ]:
# Calcular métricas para propuesta
total_distritos = df_final.count()
distritos_riesgo = df_final.filter(col("en_riesgo") == 1).count()
distritos_criticos = df_final.filter(col("grad_pct_clean") < 70).count()

df_pov = df_final.filter(col("Poverty").isNotNull())
corr_pov = df_pov.stat.corr("Poverty", "grad_pct_clean")
corr_child = df_pov.stat.corr("ChildPoverty", "grad_pct_clean")
corr_drop = df_final.stat.corr("dropout_pct_clean", "grad_pct_clean")

print("="*70)
print("💡 PROPUESTA DE VALOR")
print("="*70)

propuesta = f"""
┌──────────────────────────────────────────────────────────────────────┐
│   SISTEMA DE ALERTA TEMPRANA PARA DISTRITOS ESCOLARES EN RIESGO     │
├──────────────────────────────────────────────────────────────────────┤
│                                                                      │
│  🎯 PROBLEMA IDENTIFICADO:                                           │
│     • {distritos_riesgo} distritos con graduación < 80%                      │
│     • {distritos_criticos} distritos en situación CRÍTICA (< 70%)                │
│     • Fuerte correlación pobreza-bajo rendimiento                    │
│                                                                      │
│  💡 SOLUCIÓN PROPUESTA:                                              │
│     Dashboard predictivo que integra datos educativos +              │
│     socioeconómicos para identificar distritos en riesgo             │
│     ANTES de que sea tarde.                                          │
│                                                                      │
│  ✅ BENEFICIOS:                                                      │
│     1. Priorización de recursos basada en datos                      │
│     2. Intervención temprana en distritos vulnerables                │
│     3. Reducción de brechas educativas por nivel socioeconómico      │
│     4. Mejora en tasas de graduación estatal                         │
│                                                                      │
│  👥 USUARIOS OBJETIVO:                                               │
│     • Departamento de Educación del Estado de NY                     │
│     • Superintendentes de distritos escolares                        │
│     • Investigadores en políticas educativas                         │
│                                                                      │
│  📊 KPIs PROPUESTOS:                                                 │
│     • Reducir distritos en riesgo en 20% en 3 años                   │
│     • Aumentar graduación promedio estatal de 87% a 92%              │
│     • Cerrar brecha de graduación por pobreza en 50%                 │
│                                                                      │
└──────────────────────────────────────────────────────────────────────┘
"""

print(propuesta)

---
## 10. Conclusiones y Recomendaciones

In [ ]:
print("="*70)
print("📋 CONCLUSIONES Y RECOMENDACIONES")
print("="*70)

conclusiones = f"""
═══════════════════════════════════════════════════════════════════════
                          HALLAZGOS CLAVE
═══════════════════════════════════════════════════════════════════════

1️⃣ MAGNITUD DEL PROBLEMA:
   • {distritos_riesgo} de {total_distritos} distritos en riesgo ({distritos_riesgo/total_distritos*100:.1f}%)
   • {distritos_criticos} distritos en estado CRÍTICO (graduación < 70%)

2️⃣ FACTORES DE RIESGO PRINCIPALES:
   • Alta deserción escolar (corr: {corr_drop:.3f})
   • Pobreza del condado (corr: {corr_pov:.3f})
   • Pobreza infantil (corr: {corr_child:.3f})

3️⃣ DISPARIDADES GEOGRÁFICAS:
   • NYC tiene perfil diferente al resto del estado
   • Condados rurales con alta pobreza son más vulnerables

4️⃣ VALOR DEL DATASET COMPLEMENTARIO:
   • Contextualiza el rendimiento educativo
   • Identifica factores socioeconómicos subyacentes
   • Mejora capacidad predictiva

═══════════════════════════════════════════════════════════════════════
                         RECOMENDACIONES
═══════════════════════════════════════════════════════════════════════

→ Implementar monitoreo continuo de distritos críticos
→ Priorizar inversión en condados con alta pobreza infantil
→ Desarrollar programas de retención para reducir deserción
→ Crear incentivos para distritos que mejoren sus métricas
→ Establecer colaboración entre distritos exitosos y en riesgo

═══════════════════════════════════════════════════════════════════════
                         PRÓXIMOS PASOS
═══════════════════════════════════════════════════════════════════════

1. Desarrollar modelo ML predictivo para clasificación
2. Crear dashboard interactivo con QuickSight/Tableau
3. Implementar pipeline de actualización automática
4. Validar resultados con expertos en educación
"""

print(conclusiones)

In [ ]:
# Guardar resultados en S3
print("\n💾 GUARDANDO RESULTADOS EN S3")
print("="*60)

# Guardar como CSV
df_final.coalesce(1).write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(f"{PATH_OUTPUT}merged_data_csv")

print(f"✅ CSV guardado en: {PATH_OUTPUT}merged_data_csv")

# Guardar como Parquet (mejor para Spark)
df_final.write \
    .mode("overwrite") \
    .parquet(f"{PATH_OUTPUT}merged_data_parquet")

print(f"✅ Parquet guardado en: {PATH_OUTPUT}merged_data_parquet")

In [ ]:
print("\n" + "="*70)
print("✅ PROYECTO COMPLETADO EXITOSAMENTE EN AMAZON EMR")
print("="*70)
print(f"""
📊 Resumen:
   • Registros procesados: {df_final.count():,}
   • Distritos analizados: {df_final.select('lea_name').distinct().count()}
   • Distritos en riesgo: {distritos_riesgo}
   • Datos guardados en: {PATH_OUTPUT}
""")